# Lab: Multi-Agent Collaboration with Azure OpenAI and LangChain
## Lab Overview: What this lab is and why it matters
This lab builds a **production-grade multi-agent collaboration system** using Azure OpenAI, LangChain, and LangGraph. A multi-agent system is not simply several agents running independently — it is a coordinated architecture where agents with specialised capabilities communicate, delegate, share state, and synthesise results in ways that no single agent could achieve reliably alone.

The lab is grounded in five theoretical areas drawn directly from the ReAct and LangGraph loop primitives: the ReAct Loop and Agent State, Shared Memory and Message Passing, Supervisor and Handoff Patterns, Human-in-the-Loop Control, and Fault Tolerance with Retry Logic. Each theory section uses the multi-agent system being built as its concrete example, so the theory is never abstract.

The system built across this lab models a **cross-functional investment and clinical risk committee**: a Supervisor agent receives a complex query, routes sub-tasks to a Finance Analyst agent and a Clinical Risk agent, collects their specialist outputs, and synthesises a final committee report. This scenario requires all five collaboration patterns and produces an auditable reasoning trace suitable for regulated industries.
## What You Will Learn: Skills and understanding this lab builds
By completing this lab you will be able to:

1. Explain the ReAct loop in a multi-agent context and trace how agent state flows between nodes.
2. Implement shared memory using a common checkpointer and pass structured messages between agents.
3. Build a supervisor agent that decomposes queries, routes to specialists, and synthesises their outputs.
4. Implement a handoff pattern where one agent transfers control to another with explicit context.
5. Insert human-in-the-loop checkpoints that pause execution for review before proceeding.
6. Add fault tolerance: tool-level retries, agent-level fallbacks, and graceful degradation when a specialist cannot complete its task.
## Prerequisites: What you need before starting
- The Single-Agent Baseline lab completed. All patterns here build on `create_react_agent`, `MemorySaver`, threading, and tool invocation patterns from that lab.
- An Azure OpenAI resource with GPT-4o deployed. The same credentials apply.
- Python 3.10 or later. Python 3.12 is recommended.
- A Jupyter-compatible environment.

## Section 1: Theory — The ReAct Loop in a Multi-Agent Context
### 1.1 What the ReAct Loop Is
**ReAct** (Reason + Act) is the execution pattern that drives every agent in this lab. In a single-agent system, the loop runs within one graph: the model reasons, selects a tool, the `ToolNode` executes it, the result is appended to the message list, and the model reasons again. This continues until the model emits a final answer with no pending tool calls.

In a multi-agent system, the ReAct loop does not disappear — it is nested and federated. Each agent runs its own internal ReAct loop. The supervisor's loop produces outputs that become inputs to a specialist's loop. The specialist's loop produces outputs that return to the supervisor's loop. The multi-agent graph is a graph of loops, not a replacement for them.

In the investment committee system built in this lab, the Supervisor runs one ReAct loop. When it decides to delegate to the Finance Analyst, it emits a delegation message. The Finance Analyst runs its own ReAct loop — fetching prices, calculating values, retrieving fundamentals — and returns a structured report. The Supervisor's loop then resumes with that report as new context.
### 1.2 Agent State in a Multi-Agent Graph
State in LangGraph is a typed dictionary that flows through every node. The default state schema for `create_react_agent` is `MessagesState`, which contains a single key: `messages`, a list of `BaseMessage` objects.

In a multi-agent graph, state management has two layers. The **inner state** is the message list inside each specialist agent's own graph — visible only to that agent. The **outer state** is the message list in the supervisor's graph, which receives the specialist's final output as a new message appended to the outer conversation.

This two-layer model is the reason multi-agent systems are composable. Each specialist is a black box from the supervisor's perspective: it receives a query string, it returns a result string. The internal reasoning steps are encapsulated. In an audit context, you can inspect them (the inner state is stored separately by the specialist's checkpointer), but the supervisor does not need to reason about them.
### 1.3 Why Token Efficiency Matters More in Multi-Agent Systems
Every agent in the system sends its own message history to the model with each turn. A supervisor with two specialists, each running a 5-turn ReAct loop internally, can consume 6 to 10 times the tokens of a single agent answering the same query. This is the fundamental cost of coordination.

Three practices mitigate this. First, specialists return compact structured summaries, not raw tool outputs. Second, the supervisor passes only the relevant context when delegating, not its entire message history. Third, each agent is given a strict `recursion_limit` to prevent runaway loops from compounding the cost.

## Section 2: Theory — Shared Memory and Message Passing
### 2.1 Shared Memory in a Multi-Agent System
In the single-agent baseline, memory means one checkpointer keyed to one `thread_id`. In a multi-agent system, memory has three distinct scopes that must be deliberately designed.

**Session memory** is the conversation history within the supervisor's thread. It accumulates the user's original query, the supervisor's delegation decisions, each specialist's summary output, and the final synthesised answer. This is the primary audit record.

**Specialist memory** is the internal reasoning trace of each specialist agent, stored under its own `thread_id`. It contains every tool call and result the specialist made to produce its summary. In regulated industries, this trace is the evidence trail behind the summary.

**Shared workspace** is a data structure passed explicitly between agents — not via the checkpointer but as content in a message. In this lab, the supervisor passes a scoped task query to each specialist. The specialist reads it, performs its analysis, and returns a structured report. The supervisor merges the reports into the final output.
### 2.2 Message Passing Between Agents
Agents communicate through messages. A message is not just a string. It has a type (`HumanMessage`, `AIMessage`, `ToolMessage`), a sender identity, and a content payload. In a multi-agent graph, the message type signals the routing logic.

In the investment committee system, the supervisor sends delegation messages as `HumanMessage` objects directed at each specialist. The specialist receives them as user input to its own ReAct loop. The specialist's final response is captured and injected back into the supervisor's message list as a `ToolMessage`, so the supervisor treats specialist output the same way it treats tool output — as new evidence to reason over.

This design is deliberate. It means the supervisor's reasoning pattern does not change whether the new information came from a data API or from a specialist agent. The abstraction is consistent.
### 2.3 Avoiding Context Contamination
A common failure mode in multi-agent systems is context contamination: a specialist is given too much of the supervisor's history and begins reasoning about things outside its scope. The Finance Analyst should not be reasoning about clinical guidelines. The Clinical Risk agent should not be reasoning about portfolio weights.

The fix is **query scoping at the delegation boundary**. When the supervisor delegates, it constructs a targeted query that contains only the information the specialist needs, not a dump of the full conversation. This is implemented in Step 9 of this lab.

## Section 3: Theory — Supervisor and Handoff Patterns
### 3.1 The Supervisor Pattern
The **supervisor pattern** is the most common multi-agent topology in enterprise applications. One orchestrator agent receives all user input. It analyses the query, determines which specialists are needed, delegates to them in the appropriate sequence or in parallel, and synthesises their outputs into a final response.

The supervisor does not perform specialist tasks directly. Its role is decomposition, routing, and synthesis. A well-designed supervisor prompt contains three elements: a description of its coordination role, a description of each available specialist and when to use it, and instructions for how to structure the final synthesised output.

In the investment committee system, the supervisor receives queries like "Assess PFE as both a financial investment and a clinical risk stakeholder." It identifies that this requires a financial analysis (price, fundamentals, yield) and a clinical risk analysis (drug pipeline, regulatory exposure). It delegates both, waits for responses, and writes the committee report.
### 3.2 The Handoff Pattern
A **handoff** is a directed transfer of control from one agent to another, where the receiving agent is expected to complete a task that the originating agent has started or partially scoped. Unlike a supervisor delegation (where the supervisor retains overall control), a handoff temporarily transfers primary responsibility.

Handoffs are used when task completion requires domain expertise that the originating agent lacks and the handoff destination has full authority to determine how to proceed. In the clinical scenario in this lab, the Finance Analyst hands off to the Clinical Risk agent when a patent cliff flag is raised that requires pipeline analysis — the Clinical Risk agent then decides what further analysis is needed, rather than waiting for the supervisor to specify it.

In this lab, a handoff is implemented by invoking the target agent with a structured context message that includes the triggering finding and the specific question requiring specialist authority. The receiving agent runs its full ReAct loop autonomously and returns a complete response.
### 3.3 Deciding Between Supervisor and Handoff
The practical distinction: use the supervisor pattern when the orchestrator should remain in control of the task decomposition at each step. Use the handoff pattern when one agent needs to fully yield a sub-task to another and trust it to determine its own approach. Most enterprise systems use both — the supervisor controls the top-level workflow; handoffs handle domain escalations within specialist domains.

## Section 4: Theory — Human-in-the-Loop Control
### 4.1 Why Human-in-the-Loop Is Non-Negotiable in Regulated Industries
In financial services and healthcare, an agent that acts autonomously without any human checkpoint is not a productivity tool — it is a liability. MiFID II, the FCA's AI guidance, and FDA guidance on clinical decision support all require that consequential decisions have a documented human review step.

Human-in-the-loop (HITL) control does not mean a human reviews every token. It means that at defined decision gates — before a recommendation is committed to a client record, before a drug interaction flag is acted on, before a trade is executed — execution pauses and a human confirms or rejects before the system continues.

LangGraph implements HITL through **interrupt points**: nodes in the graph where execution is suspended, the current state is persisted to the checkpointer, and control is returned to the calling application. The human reviewer reads the state, optionally modifies it, and resumes execution. The graph continues from exactly where it paused.
### 4.2 Interrupt Points in LangGraph
An interrupt is called inside a node using `interrupt()`. When execution reaches that call, LangGraph suspends the graph, persists state to the checkpointer, and raises a `GraphInterrupt` exception that the caller catches. The human reviewer reads the interrupted state, provides input, and resumes via `graph.invoke(Command(resume=human_input), config=same_thread_config)`.

In the investment committee system, the interrupt is placed in the `human_review` node — after both specialists have completed but before synthesis. The human reviewer reads both specialist reports and can add a directive that the synthesis step incorporates.

State persistence during an interrupt is handled by the same `MemorySaver` checkpointer used for threading. The interrupt is a long pause in thread execution. When the human resumes, the `thread_id` reloads the exact state from the pause point.
### 4.3 HITL and Audit Trails
Every interrupt creates a checkpoint in the state history. The audit trail is not just the final output — it includes what state was presented for review, what the human modified, and when resumption occurred. This is a stronger audit record than a log line because the complete reasoning state at the point of review is preserved and reproducible.

## Section 5: Theory — Fault Tolerance and Retry Logic
### 5.1 Why Multi-Agent Systems Fail Across More Dimensions Than Single Agents
A single agent has one failure mode at the tool level: a tool returns an error or unexpected output. A multi-agent system compounds this. A specialist can fail to complete its task. A network call to an external data source can time out. The supervisor can misroute a query to the wrong specialist. A specialist can exceed its recursion limit. The synthesis step can receive incomplete inputs from one specialist and still need to produce a useful output.

Fault tolerance in a multi-agent system is therefore a multi-layer concern: tool-level retries within specialists, agent-level fallbacks at the supervisor, and graceful degradation in the synthesis step when some specialist outputs are missing.
### 5.2 Tool-Level Retry Pattern
Individual tools can fail for transient reasons: a data feed is temporarily unavailable, an external API returns a rate-limit error. The standard pattern is a retry wrapper with a fixed number of attempts and a short delay between them. The retry should be applied at the tool level, not the agent level. Retrying an entire agent turn because one tool failed wastes tokens and increases latency. Retrying just the tool call is targeted and efficient.
### 5.3 Agent-Level Fallback and Graceful Degradation
When a specialist fails entirely — either because all retries are exhausted or because the query is outside its scope — the supervisor must have a defined fallback behaviour. The two options are: substitute a default response (a data-unavailable notice or a conservative safe-harbour statement) or escalate to human review.

In the investment committee system, the fallback for the Finance Analyst is a conservative statement: financial data unavailable, recommend deferring. The fallback for the Clinical Risk agent is an escalation: if clinical data cannot be retrieved, the system does not synthesise a report without it — it interrupts for human review.

Graceful degradation means the system produces a useful, clearly caveated output rather than crashing or producing a silently incomplete one. A report that says "Clinical risk data was unavailable; financial analysis only" is more useful and safer than a report that omits the clinical section without explanation.

## Step 1: Install and Verify Packages
The package set is identical to the baseline lab. Run this cell to confirm version consistency before building the multi-agent graph.

In [1]:
%pip install \
    langchain \
    langchain-openai \
    langchain-community \
    langgraph \
    openai \
    python-dotenv \
    tiktoken \
    --upgrade --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.5/167.5 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.0/503.0 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import importlib, sys

print(f"Python: {sys.version}")
print()

for pkg in ["langchain_core", "langchain", "langchain_openai", "langgraph", "openai"]:
    try:
        m = importlib.import_module(pkg)
        print(f"{pkg:<30} {getattr(m, '__version__', 'unknown')}")
    except ImportError:
        print(f"{pkg:<30} NOT INSTALLED")

print()
checks = [
    ("langgraph.prebuilt",          "create_react_agent"),
    ("langgraph.checkpoint.memory", "MemorySaver"),
    ("langchain_openai",             "AzureChatOpenAI"),
    ("langchain_core.tools",         "tool"),
    ("langchain_core.messages",      "HumanMessage"),
    ("langgraph.graph",              "StateGraph"),
    ("langgraph.types",              "Command"),
]
all_ok = True
for module_path, symbol in checks:
    try:
        getattr(importlib.import_module(module_path), symbol)
        print(f"  OK   from {module_path} import {symbol}")
    except (ImportError, AttributeError) as e:
        print(f"  FAIL from {module_path} import {symbol} -> {e}")
        all_ok = False

print()
print("All imports healthy." if all_ok else "Fix: re-run the install cell, restart kernel, retry.")

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]

langchain_core                 1.2.18
langchain                      1.2.12
langchain_openai               unknown
langgraph                      unknown
openai                         2.26.0

  OK   from langgraph.prebuilt import create_react_agent
  OK   from langgraph.checkpoint.memory import MemorySaver
  OK   from langchain_openai import AzureChatOpenAI
  OK   from langchain_core.tools import tool
  OK   from langchain_core.messages import HumanMessage
  OK   from langgraph.graph import StateGraph
  OK   from langgraph.types import Command

All imports healthy.


## Step 2: Configure Credentials and Shared Imports
Credentials are loaded from the same `.env` file as the baseline lab. The imports block here adds the LangGraph primitives (`StateGraph`, `MessagesState`, `Command`, `interrupt`) needed for the custom committee graph and HITL patterns.

In [3]:
from pathlib import Path

env_content = """
AZURE_OPENAI_ENDPOINT=https://<TBD>.cognitiveservices.azure.com
AZURE_OPENAI_API_KEY=
AZURE_OPENAI_DEPLOYMENT_NAME=gpt-4o
AZURE_OPENAI_API_VERSION=2024-12-01-preview
"""

# Create .env file in current directory
env_path = Path(".env")
env_path.write_text(env_content.strip())

print(".env file created successfully.")
print(env_path.resolve())

.env file created successfully.
/content/.env


In [4]:
import os, uuid, time
from dotenv import load_dotenv

load_dotenv(override=True)

required = [
    "AZURE_OPENAI_API_KEY",
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_DEPLOYMENT_NAME",
    "AZURE_OPENAI_API_VERSION",
]
missing = [v for v in required if not os.getenv(v)]
if missing:
    raise EnvironmentError(f"Missing variables: {missing}")

print("Credentials loaded.")
print(f"  Endpoint:   {os.getenv('AZURE_OPENAI_ENDPOINT')}")
print(f"  Deployment: {os.getenv('AZURE_OPENAI_DEPLOYMENT_NAME')}")

Credentials loaded.
  Endpoint:   https://<TBD>.cognitiveservices.azure.com
  Deployment: gpt-4o


In [5]:
from langchain_openai import AzureChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.types import Command, interrupt
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import concurrent.futures

llm = AzureChatOpenAI(
    azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    temperature=0,
    max_tokens=4096,
)

resp = llm.invoke([HumanMessage(content="Respond with exactly: READY")])
print(f"Model connectivity: {resp.content.strip()}")

Model connectivity: READY


## Step 3: Define Finance Tools — Finance Analyst Specialist
Finance tools are used exclusively by the Finance Analyst agent. The Supervisor has no tools of its own — its capability is reasoning, routing, and synthesis. Equipping the supervisor with data tools blurs the architectural boundary and degrades routing quality because the supervisor begins retrieving data instead of delegating.

In [6]:
@tool
def get_stock_price(ticker: str) -> str:
    """Retrieve current market price for an equity ticker. Returns price in USD."""
    prices = {"AAPL": 189.45, "MSFT": 415.20, "JPM": 198.73,
              "GS": 452.10, "JNJ": 162.88, "PFE": 29.54, "BAC": 38.12,
              "MRK": 128.75, "ABBV": 175.30}
    t = ticker.upper().strip()
    p = prices.get(t)
    return f"{t}: ${p:.2f} USD" if p else f"Ticker {t} not found in data feed."


@tool
def get_company_fundamentals(ticker: str) -> str:
    """Retrieve P/E ratio, market cap, dividend yield, sector, and beta for a ticker."""
    data = {
        "JNJ":  {"pe": 15.8, "cap": "390B",  "div": "3.10%", "sector": "Healthcare", "beta": 0.54},
        "PFE":  {"pe": 9.1,  "cap": "165B",  "div": "6.20%", "sector": "Healthcare", "beta": 0.62},
        "MRK":  {"pe": 18.2, "cap": "325B",  "div": "2.55%", "sector": "Healthcare", "beta": 0.48},
        "ABBV": {"pe": 21.4, "cap": "310B",  "div": "3.85%", "sector": "Healthcare", "beta": 0.71},
        "JPM":  {"pe": 11.2, "cap": "572B",  "div": "2.30%", "sector": "Financials", "beta": 1.12},
        "GS":   {"pe": 13.5, "cap": "145B",  "div": "2.65%", "sector": "Financials", "beta": 1.38},
    }
    t = ticker.upper().strip()
    f = data.get(t)
    if not f:
        return f"Fundamentals not available for {t}."
    return (f"{t} | P/E: {f['pe']} | Cap: {f['cap']} | Div Yield: {f['div']} "
            f"| Sector: {f['sector']} | Beta: {f['beta']}")


@tool
def get_rate(rate_type: str) -> str:
    """Look up a benchmark rate. Supported: fed_funds, sofr, us_10yr, us_2yr, us_30yr."""
    rates = {"fed_funds": "5.25%", "sofr": "5.30%",
             "us_10yr": "4.48%", "us_2yr": "4.85%", "us_30yr": "4.62%"}
    key = rate_type.lower().strip()
    r = rates.get(key)
    return f"{rate_type.replace('_',' ').title()}: {r}" if r else \
           f"Rate '{rate_type}' not recognised. Options: {', '.join(rates.keys())}."


@tool
def calculate_portfolio_value(holdings: str) -> str:
    """Calculate total market value from comma-separated TICKER:SHARES pairs. Example: 'PFE:500,JNJ:200'."""
    prices = {"AAPL": 189.45, "MSFT": 415.20, "JPM": 198.73,
              "GS": 452.10, "JNJ": 162.88, "PFE": 29.54, "BAC": 38.12,
              "MRK": 128.75, "ABBV": 175.30}
    total, lines = 0.0, []
    try:
        for pair in holdings.split(","):
            ticker, shares_str = pair.strip().split(":")
            t, s = ticker.strip().upper(), float(shares_str.strip())
            p = prices.get(t, 0.0)
            v = p * s
            total += v
            lines.append(f"  {t}: {s:.0f} x ${p:.2f} = ${v:,.2f}")
    except ValueError as e:
        return f"Parse error: {e}. Format: 'TICKER:SHARES,TICKER:SHARES'"
    return "Breakdown:\n" + "\n".join(lines) + f"\nTotal: ${total:,.2f} USD"


finance_tools = [get_stock_price, get_company_fundamentals, get_rate, calculate_portfolio_value]
print(f"Finance tools defined: {[t.name for t in finance_tools]}")

Finance tools defined: ['get_stock_price', 'get_company_fundamentals', 'get_rate', 'calculate_portfolio_value']


## Step 4: Define Clinical Tools — Clinical Risk Specialist
Clinical tools serve the Clinical Risk agent exclusively. Note `assess_pipeline_risk` — this tool has no equivalent in the single-agent baseline. It encodes pharmaceutical-specific domain knowledge that belongs in the clinical specialist, not in a general-purpose agent. This is the value of specialisation: each agent can carry deep domain tools without polluting the other's tool selection.

In [7]:
@tool
def check_drug_interaction(drug_a: str, drug_b: str) -> str:
    """Check for a clinical drug interaction between two medications. Returns severity and clinical note."""
    db = {
        frozenset(["warfarin",     "aspirin"]):         ("MAJOR",    "Elevated bleeding risk. Monitor INR closely."),
        frozenset(["metformin",    "ibuprofen"]):        ("MODERATE", "NSAIDs reduce metformin clearance. Risk of lactic acidosis."),
        frozenset(["lisinopril",   "potassium"]):        ("MODERATE", "ACE inhibitors raise serum potassium. Avoid supplements."),
        frozenset(["atorvastatin", "clarithromycin"]):   ("MAJOR",    "CYP3A4 inhibition raises statin levels. Suspend atorvastatin."),
        frozenset(["clopidogrel",  "omeprazole"]):       ("MODERATE", "CYP2C19 inhibition reduces clopidogrel activation."),
        frozenset(["warfarin",     "clarithromycin"]):   ("MAJOR",    "CYP2C9 inhibition elevates warfarin anticoagulation. INR monitoring required."),
    }
    key = frozenset([drug_a.lower().strip(), drug_b.lower().strip()])
    result = db.get(key)
    if result:
        sev, note = result
        return f"Severity: {sev} | {note}"
    return f"No major interaction documented between {drug_a} and {drug_b}."


@tool
def get_clinical_guideline(condition: str) -> str:
    """Retrieve evidence-based management guideline. Supported: type2_diabetes, hypertension, atrial_fibrillation, heart_failure."""
    guidelines = {
        "type2_diabetes":      "ADA 2024: First-line Metformin + lifestyle (HbA1c <7.0%). Second-line GLP-1 or SGLT-2 based on comorbidity.",
        "hypertension":        "ACC/AHA 2017: Target <130/80 mmHg. First-line ACE inhibitor, ARB, thiazide, or CCB.",
        "atrial_fibrillation": "ESC 2020: CHA2DS2-VASc >=2 (male): anticoagulate. Prefer NOACs over warfarin unless contraindicated.",
        "heart_failure":       "ESC 2021 HFrEF: ACE inhibitor/ARNi + beta-blocker + MRA + SGLT-2. ICD if LVEF <=35% after 3 months optimal therapy.",
    }
    key = condition.lower().strip().replace(" ", "_")
    g = guidelines.get(key)
    return g if g else f"No guideline for '{condition}'. Supported: {', '.join(guidelines.keys())}."


@tool
def assess_pipeline_risk(company: str) -> str:
    """Assess clinical pipeline and regulatory risk for a pharmaceutical company. Returns pipeline summary and risk flags."""
    pipeline = {
        "PFE": (
            "Pfizer: Danuglipron (GLP-1/obesity, Phase 3, PDUFA 2025). Elranatamab (multiple myeloma, approved 2023). "
            "Risk flags: Post-COVID revenue normalisation; Eliquis patent cliff 2026 (~$6B annual revenue)."
        ),
        "JNJ": (
            "J&J: Talvey (multiple myeloma, approved 2023). Rybrevant (NSCLC, expanding indications). "
            "Risk flags: Kenvue spin-off narrows to pharma/medtech; Stelara biosimilar competition 2025; talc litigation."
        ),
        "MRK": (
            "Merck: Keytruda (21 approved indications, $25B+ 2023 revenue, patent 2028). "
            "Risk flags: Heavy Keytruda concentration; IRA pricing negotiation impact on Januvia."
        ),
        "ABBV": (
            "AbbVie: Skyrizi + Rinvoq offsetting Humira biosimilar erosion (lost exclusivity Jan 2023). "
            "Risk flags: Humira transition pace uncertain; Botox aesthetic demand sensitivity."
        ),
    }
    key = company.upper().strip()
    p = pipeline.get(key)
    return p if p else f"Pipeline data not available for {key}. Supported: {', '.join(pipeline.keys())}."


@tool
def get_lab_reference_range(test_name: str) -> str:
    """Return the standard adult reference range for a laboratory test. Supported: hba1c, creatinine, potassium, inr, haemoglobin, ldl."""
    ranges = {
        "hba1c":       "4.0-5.6% normal; 5.7-6.4% pre-diabetes; >=6.5% diabetes. Target <7.0% in managed T2DM.",
        "creatinine":  "Male 0.74-1.35 mg/dL; Female 0.59-1.04 mg/dL. Elevated: renal impairment.",
        "potassium":   "3.5-5.1 mEq/L. <3.5 hypokalaemia; >5.5 hyperkalaemia (cardiac risk).",
        "inr":         "Therapeutic warfarin range: 2.0-3.0 general; 2.5-3.5 mechanical heart valves.",
        "haemoglobin": "Male 13.5-17.5 g/dL; Female 12.0-15.5 g/dL.",
        "ldl":         "Optimal <100 mg/dL; High >=160; Very high >=190. Target <70 in very high CV risk.",
    }
    key = test_name.lower().strip()
    r = ranges.get(key)
    return r if r else f"Reference range not available for '{test_name}'. Supported: {', '.join(ranges.keys())}."


clinical_tools = [check_drug_interaction, get_clinical_guideline, assess_pipeline_risk, get_lab_reference_range]
print(f"Clinical tools defined: {[t.name for t in clinical_tools]}")

Clinical tools defined: ['check_drug_interaction', 'get_clinical_guideline', 'assess_pipeline_risk', 'get_lab_reference_range']


## Step 5: Create the Two Specialist Agents
Each specialist is a complete `create_react_agent` instance with its own system prompt, tool set, and checkpointer. The checkpointers are separate: each specialist's internal reasoning trace is stored under its own thread namespace. This is the two-layer memory model described in Section 2.1 — specialist memory is isolated from session memory.

In [8]:
FINANCE_ANALYST_PROMPT = (
    "You are a Finance Analyst specialist in a cross-functional investment committee. "
    "Your scope is strictly financial: market prices, valuation multiples, dividend yields, "
    "benchmark rates, portfolio values, and capital allocation recommendations. "
    "Do not comment on clinical, regulatory, or medical topics. "
    "Always use tools to retrieve current data. Present figures with units. "
    "Conclude with a structured FINANCIAL SUMMARY block: Price, P/E, Dividend Yield, Beta, investment view. "
    "This output is informational only and does not constitute investment advice."
)

CLINICAL_RISK_PROMPT = (
    "You are a Clinical Risk specialist in a cross-functional investment committee. "
    "Your scope is clinical and regulatory: drug pipelines, patent cliffs, regulatory risks, "
    "drug interactions, clinical guidelines, and patient safety considerations. "
    "Do not comment on financial valuation, share prices, or capital markets. "
    "Always use tools to retrieve current pipeline and clinical data. "
    "Conclude with a structured CLINICAL RISK SUMMARY block: Pipeline Stage, Key Risk Flags, risk assessment. "
    "This output is informational only and requires review by a qualified clinician."
)

finance_analyst = create_react_agent(
    model=llm,
    tools=finance_tools,
    prompt=FINANCE_ANALYST_PROMPT,
    checkpointer=MemorySaver(),
)

clinical_risk = create_react_agent(
    model=llm,
    tools=clinical_tools,
    prompt=CLINICAL_RISK_PROMPT,
    checkpointer=MemorySaver(),
)

print("Finance Analyst nodes:  ", list(finance_analyst.nodes.keys()))
print("Clinical Risk nodes:    ", list(clinical_risk.nodes.keys()))

Finance Analyst nodes:   ['__start__', 'agent', 'tools']
Clinical Risk nodes:     ['__start__', 'agent', 'tools']


/tmp/ipykernel_160/3319844070.py:21: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  finance_analyst = create_react_agent(
/tmp/ipykernel_160/3319844070.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  clinical_risk = create_react_agent(


## Step 6: Build Invocation and Tracing Utilities
These utilities are used throughout the lab. `invoke_specialist` runs a specialist with a scoped query under its own `thread_id`, capturing the inner ReAct loop steps and token usage. The `thread_id` returned is the key to the specialist's checkpointer — it can be used to retrieve the full inner trace for audit purposes.

In [9]:
def invoke_specialist(agent, query: str, agent_name: str = "Specialist") -> dict:
    """Invoke a specialist agent with a scoped query. Returns answer, thread_id, token_usage, steps."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}, "recursion_limit": 12}
    result = agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=config
    )
    messages = result["messages"]
    answer = messages[-1].content
    total_in, total_out, steps = 0, 0, []
    for msg in messages:
        usage = getattr(msg, "response_metadata", {}).get("token_usage", {})
        if usage:
            total_in  += usage.get("prompt_tokens", 0)
            total_out += usage.get("completion_tokens", 0)
        tool_names = [tc["name"] for tc in getattr(msg, "tool_calls", [])]
        steps.append({"type": type(msg).__name__, "tools": tool_names})
    return {
        "agent":       agent_name,
        "thread_id":   thread_id,
        "answer":      answer,
        "token_usage": {"in": total_in, "out": total_out, "total": total_in + total_out},
        "steps":       steps,
    }


def print_specialist_trace(result: dict) -> None:
    """Print the step-level inner ReAct loop trace for a specialist invocation."""
    print(f"{'─'*65}")
    print(f"  Agent:    {result['agent']}")
    print(f"  ThreadID: {result['thread_id'][:20]}...")
    print(f"  Tokens:   in={result['token_usage']['in']}  out={result['token_usage']['out']}  total={result['token_usage']['total']}")
    print(f"  Steps:")
    for i, s in enumerate(result["steps"]):
        tool_str = f" -> {s['tools']}" if s["tools"] else ""
        print(f"    {i+1}. {s['type']}{tool_str}")
    print(f"{'─'*65}")


print("Utilities defined.")

Utilities defined.


## Step 7: Validate Specialists Independently Before Composing
A multi-agent system cannot be reliable if its components are not first validated independently. This step runs each specialist in isolation on a known query. Only after both pass independently does the composition proceed. This is the build discipline described in Section 4 theory — reliable components first, composition second.

In [10]:
print("Validating Finance Analyst...")
print()
fa_val = invoke_specialist(
    finance_analyst,
    "Retrieve current price and full fundamentals for PFE. Also fetch the US 10-year Treasury yield.",
    agent_name="Finance Analyst"
)
print_specialist_trace(fa_val)
print()
print(fa_val["answer"])

Validating Finance Analyst...

─────────────────────────────────────────────────────────────────
  Agent:    Finance Analyst
  ThreadID: 1e7e204c-ab26-4c9c-9...
  Tokens:   in=733  out=293  total=1026
  Steps:
    1. HumanMessage
    2. AIMessage -> ['get_stock_price', 'get_company_fundamentals', 'get_rate']
    3. ToolMessage
    4. ToolMessage
    5. ToolMessage
    6. AIMessage
─────────────────────────────────────────────────────────────────

### Financial Data for Pfizer (PFE) and US 10-Year Treasury Yield:

1. **Pfizer (PFE):**
   - **Current Price:** $29.54 USD
   - **P/E Ratio:** 9.1
   - **Market Capitalization:** $165 billion USD
   - **Dividend Yield:** 6.20%
   - **Sector:** Healthcare
   - **Beta:** 0.62

2. **US 10-Year Treasury Yield:** 4.48%

---

### FINANCIAL SUMMARY:
- **Price:** $29.54 USD
- **P/E Ratio:** 9.1
- **Dividend Yield:** 6.20%
- **Beta:** 0.62
- **Investment View:** Pfizer offers a high dividend yield and trades at a relatively low P/E ratio, indicating p

In [11]:
print("Validating Clinical Risk agent...")
print()
cr_val = invoke_specialist(
    clinical_risk,
    "Assess the clinical pipeline and key regulatory risks for PFE.",
    agent_name="Clinical Risk"
)
print_specialist_trace(cr_val)
print()
print(cr_val["answer"])

Validating Clinical Risk agent...

─────────────────────────────────────────────────────────────────
  Agent:    Clinical Risk
  ThreadID: 8c7f1876-f6d6-4847-8...
  Tokens:   in=696  out=282  total=978
  Steps:
    1. HumanMessage
    2. AIMessage -> ['assess_pipeline_risk']
    3. ToolMessage
    4. AIMessage
─────────────────────────────────────────────────────────────────

### CLINICAL RISK SUMMARY

**Pipeline Stage:**
- **Danuglipron**: GLP-1 receptor agonist for obesity, currently in Phase 3 trials. Anticipated PDUFA date in 2025.
- **Elranatamab**: Approved in 2023 for multiple myeloma.

**Key Risk Flags:**
1. **Post-COVID Revenue Normalisation**: Pfizer faces challenges in maintaining revenue levels as demand for COVID-related products declines.
2. **Eliquis Patent Cliff**: Patent expiration in 2026 for Eliquis, a major anticoagulant, poses significant revenue risks (~$6 billion annual revenue).

**Risk Assessment:**
- **Danuglipron**: GLP-1 receptor agonists are a competitive s

## Step 8: Build the Supervisor Agent
The Supervisor is itself a `create_react_agent`. Its tools are the two specialist agents, each wrapped as a `@tool`. From the supervisor's perspective, calling a specialist is identical to calling a data API: it provides inputs and receives structured output. This is the core insight of the supervisor pattern — specialisation is abstracted behind a uniform tool interface.

In [12]:
@tool
def delegate_to_finance_analyst(query: str) -> str:
    """
    Delegate a financial analysis task to the Finance Analyst specialist.
    Use for: stock prices, fundamentals, valuation multiples, dividend yields, benchmark rates, portfolio values.
    Do NOT use for clinical, pipeline, or regulatory questions.
    Returns: structured financial analysis with a FINANCIAL SUMMARY block.
    """
    result = invoke_specialist(finance_analyst, query, agent_name="Finance Analyst")
    print(f"    [inner loop] Finance Analyst | tokens={result['token_usage']['total']} | steps={len(result['steps'])}")
    return result["answer"]


@tool
def delegate_to_clinical_risk(query: str) -> str:
    """
    Delegate a clinical or regulatory risk analysis to the Clinical Risk specialist.
    Use for: drug pipeline assessments, patent cliff analysis, regulatory risk flags, drug interactions, clinical guidelines.
    Do NOT use for financial valuation, share prices, or market data.
    Returns: structured risk analysis with a CLINICAL RISK SUMMARY block.
    """
    result = invoke_specialist(clinical_risk, query, agent_name="Clinical Risk")
    print(f"    [inner loop] Clinical Risk | tokens={result['token_usage']['total']} | steps={len(result['steps'])}")
    return result["answer"]


SUPERVISOR_PROMPT = (
    "You are the Supervisor of a cross-functional investment and clinical risk committee. "
    "You have two specialists: the Finance Analyst (all financial data) and the Clinical Risk agent (all pipeline and regulatory data). "
    "Your role is to decompose the user's query, delegate each component to the correct specialist, "
    "and synthesise their outputs into a unified committee report. "
    "When delegating, pass only the information the specialist needs — do not pass your full conversation history. "
    "Your final output must be a COMMITTEE REPORT with three sections: "
    "FINANCIAL ANALYSIS, CLINICAL RISK ASSESSMENT, and COMMITTEE RECOMMENDATION. "
    "Always state that this report is informational only."
)

supervisor_agent = create_react_agent(
    model=llm,
    tools=[delegate_to_finance_analyst, delegate_to_clinical_risk],
    prompt=SUPERVISOR_PROMPT,
    checkpointer=MemorySaver(),
)

print("Supervisor agent created.")
print(f"Supervisor tools: {[t.name for t in [delegate_to_finance_analyst, delegate_to_clinical_risk]]}")
print()
print("Graph structure (Mermaid):")
print(supervisor_agent.get_graph().draw_mermaid())

Supervisor agent created.
Supervisor tools: ['delegate_to_finance_analyst', 'delegate_to_clinical_risk']

Graph structure (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



/tmp/ipykernel_160/1423807440.py:38: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  supervisor_agent = create_react_agent(


## Step 9: Run the Full Supervisor-Specialist Collaboration
The end-to-end committee workflow. The Supervisor receives a dual-domain query about PFE, decomposes it, delegates to both specialists, and synthesises the committee report. Inner loop trace lines confirm that both specialists run their independent ReAct loops. The final output demonstrates synthesis that neither specialist could produce alone — combining financial income attractiveness with pipeline-adjusted risk.

In [13]:
committee_session = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 20}

committee_query = (
    "Produce a full investment committee report on Pfizer (PFE). "
    "I need the financial profile — price, fundamentals, dividend yield, "
    "and how it compares to the 10-year Treasury as an income investment — "
    "and the clinical risk assessment including pipeline stage, patent cliff exposure, and key regulatory risks. "
    "Synthesise both into a committee recommendation."
)

print("COMMITTEE QUERY")
print("=" * 70)
print(committee_query)
print("=" * 70)
print()
print("Running — inner loop traces appear below as specialists are invoked:")
print()

result = supervisor_agent.invoke(
    {"messages": [HumanMessage(content=committee_query)]},
    config=committee_session
)

print()
print("COMMITTEE REPORT")
print("=" * 70)
print(result["messages"][-1].content)

COMMITTEE QUERY
Produce a full investment committee report on Pfizer (PFE). I need the financial profile — price, fundamentals, dividend yield, and how it compares to the 10-year Treasury as an income investment — and the clinical risk assessment including pipeline stage, patent cliff exposure, and key regulatory risks. Synthesise both into a committee recommendation.

Running — inner loop traces appear below as specialists are invoked:

    [inner loop] Finance Analyst | tokens=1076 | steps=6
    [inner loop] Clinical Risk | tokens=1433 | steps=6

COMMITTEE REPORT
### COMMITTEE REPORT: Pfizer (PFE)

#### FINANCIAL ANALYSIS
- **Current Stock Price**: $29.54 USD
- **Fundamentals**:
  - **P/E Ratio**: 9.1
  - **Market Capitalization**: $165 billion USD
  - **Dividend Yield**: 6.20%
  - **Beta**: 0.62 (low volatility compared to the market)
- **Income Investment Comparison**:
  - Pfizer's dividend yield of **6.20%** significantly exceeds the **10-year Treasury yield of 4.48%**, making it 

## Step 10: Query Scoping at the Delegation Boundary
This step demonstrates context contamination and its fix. A poorly scoped delegation passes the full cross-domain user query to each specialist. A correctly scoped delegation extracts only what the specialist needs. The contrast shows why scoping at the delegation boundary matters: an unscoped query causes the Finance Analyst to receive clinical context it will reason about incorrectly, and the Clinical Risk agent to receive financial context it should ignore.

In [14]:
mixed_query = (
    "Our portfolio holds 1000 shares of JNJ and 2000 shares of MRK. "
    "I want the total value and a comparison of their dividend yields against SOFR, "
    "AND I need the pipeline risk and patent cliff timeline for both companies. "
    "Also retrieve the heart failure guideline for disease-area context."
)

# Correctly scoped delegation queries — this is what a well-designed supervisor emits
financial_scope = (
    "Portfolio: 1000 shares of JNJ and 2000 shares of MRK. "
    "Calculate total portfolio value. Retrieve price and fundamentals for both tickers. "
    "Fetch the SOFR rate and compare each stock's dividend yield to it."
)

clinical_scope = (
    "Assess the clinical pipeline risk and patent cliff exposure for JNJ and MRK. "
    "Retrieve the heart failure management guideline for disease-area analysis."
)

print("Scoped delegation — Finance Analyst:")
fa_scoped = invoke_specialist(finance_analyst, financial_scope, "Finance Analyst (scoped)")
print_specialist_trace(fa_scoped)
print()

print("Scoped delegation — Clinical Risk:")
cr_scoped = invoke_specialist(clinical_risk, clinical_scope, "Clinical Risk (scoped)")
print_specialist_trace(cr_scoped)
print()

print("FINANCE ANALYST OUTPUT:")
print(fa_scoped["answer"])
print()
print("CLINICAL RISK OUTPUT:")
print(cr_scoped["answer"])

Scoped delegation — Finance Analyst:
─────────────────────────────────────────────────────────────────
  Agent:    Finance Analyst (scoped)
  ThreadID: 6c174dc4-32ef-461c-b...
  Tokens:   in=949  out=583  total=1532
  Steps:
    1. HumanMessage
    2. AIMessage -> ['calculate_portfolio_value', 'get_stock_price', 'get_stock_price', 'get_company_fundamentals', 'get_company_fundamentals', 'get_rate']
    3. ToolMessage
    4. ToolMessage
    5. ToolMessage
    6. ToolMessage
    7. ToolMessage
    8. ToolMessage
    9. AIMessage
─────────────────────────────────────────────────────────────────

Scoped delegation — Clinical Risk:
─────────────────────────────────────────────────────────────────
  Agent:    Clinical Risk (scoped)
  ThreadID: ede218e7-e917-4dfc-b...
  Tokens:   in=856  out=556  total=1412
  Steps:
    1. HumanMessage
    2. AIMessage -> ['assess_pipeline_risk', 'assess_pipeline_risk', 'get_clinical_guideline']
    3. ToolMessage
    4. ToolMessage
    5. ToolMessage
    6. A

## Step 11: Shared Memory — Multi-Turn Session with the Committee
The Supervisor's `MemorySaver` gives the committee persistent session memory. Follow-up questions reference the prior committee output without the user restating context. This is the outer session memory layer from Section 2.1. Specialists run fresh inner loops for each delegation — they are stateless across turns — while the Supervisor accumulates the full session history.

In [15]:
multi_turn_session = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 20}


def committee_turn(query: str, label: str = "") -> str:
    if label:
        print(f"{'─'*65}")
        print(f"  {label}")
        print(f"  User: {query[:100]}")
        print(f"{'─'*65}")
    r = supervisor_agent.invoke(
        {"messages": [HumanMessage(content=query)]},
        config=multi_turn_session
    )
    return r["messages"][-1].content


t1 = committee_turn(
    "Give me a brief financial and clinical risk profile of JNJ.",
    label="TURN 1 — establish context"
)
print(t1[:700], "...\n" if len(t1) > 700 else "\n")

─────────────────────────────────────────────────────────────────
  TURN 1 — establish context
  User: Give me a brief financial and clinical risk profile of JNJ.
─────────────────────────────────────────────────────────────────
    [inner loop] Finance Analyst | tokens=1070 | steps=5
    [inner loop] Clinical Risk | tokens=1552 | steps=6
### COMMITTEE REPORT: Johnson & Johnson (JNJ)

#### FINANCIAL ANALYSIS
- **Current Price**: $162.88 USD
- **Price-to-Earnings (P/E) Ratio**: 15.8 (Moderate valuation compared to market averages)
- **Dividend Yield**: 3.10% (Attractive for income-focused investors)
- **Market Capitalization**: $390 billion USD (Large-cap stock indicating stability)
- **Beta**: 0.54 (Low volatility, suggesting consistent performance during market fluctuations)
- **Investment View**: JNJ is a stable, income-generating stock with moderate valuation multiples and low risk. It appeals to conservative investors seeking dividend income and long-term capital preservation.

###

In [16]:
t2 = committee_turn(
    "How does its dividend yield compare to the current US 10-year Treasury? "
    "Is JNJ an attractive income position given the rate environment?",
    label="TURN 2 — follow-up references JNJ from prior turn"
)
print(t2)

─────────────────────────────────────────────────────────────────
  TURN 2 — follow-up references JNJ from prior turn
  User: How does its dividend yield compare to the current US 10-year Treasury? Is JNJ an attractive income 
─────────────────────────────────────────────────────────────────
    [inner loop] Finance Analyst | tokens=1091 | steps=5
### COMMITTEE REPORT: JNJ Dividend Yield vs. US 10-Year Treasury

#### FINANCIAL ANALYSIS
- **JNJ Dividend Yield**: 3.10%
- **US 10-Year Treasury Yield**: 4.48%
- **Comparison**:
  - JNJ's dividend yield is lower than the current US 10-year Treasury yield, making Treasuries more attractive for pure income-focused investors in the short term.
  - JNJ offers potential for dividend growth over time, whereas Treasury yields are fixed.
- **Risk Profile**:
  - JNJ has a beta of 0.54, indicating lower volatility compared to the broader market, but it carries equity risk unlike the risk-free nature of Treasuries.
- **Valuation**: JNJ's P/E ratio of 1

In [17]:
t3 = committee_turn(
    "Now add MRK. Compare MRK's pipeline risk to JNJ's and give the committee a relative ranking "
    "of the two on both financial and clinical risk dimensions.",
    label="TURN 3 — extend to MRK, comparative synthesis"
)
print(t3)

─────────────────────────────────────────────────────────────────
  TURN 3 — extend to MRK, comparative synthesis
  User: Now add MRK. Compare MRK's pipeline risk to JNJ's and give the committee a relative ranking of the t
─────────────────────────────────────────────────────────────────
    [inner loop] Finance Analyst | tokens=1050 | steps=5
    [inner loop] Clinical Risk | tokens=1600 | steps=6
### COMMITTEE REPORT: Comparative Analysis of JNJ and MRK

#### FINANCIAL ANALYSIS
**Johnson & Johnson (JNJ)**:
- **Price**: $162.88 USD
- **P/E Ratio**: 15.8 (Moderate valuation)
- **Dividend Yield**: 3.10% (Higher yield compared to MRK)
- **Beta**: 0.54 (Low volatility)
- **Market Cap**: $390 billion USD (Larger than MRK)
- **Investment View**: Stable, income-generating stock with moderate valuation multiples and low risk. Suitable for conservative investors seeking dividend income and long-term capital preservation.

**Merck (MRK)**:
- **Price**: $128.75 USD
- **P/E Ratio**: 18.2 (Higher v

## Step 12: The Handoff Pattern — Clinical Escalation
This step implements the handoff from Section 3.2. The Finance Analyst surfaces a material clinical flag during its analysis — the Eliquis patent cliff. Rather than returning to the supervisor and waiting for another delegation cycle, it initiates a direct handoff to the Clinical Risk agent. The receiving agent is given a structured context message (originating agent, trigger finding, and specific question), then runs its own ReAct loop with full autonomy.

In [18]:
def execute_handoff(
    from_agent_name: str,
    to_agent,
    to_agent_name: str,
    trigger_finding: str,
    handoff_question: str,
) -> dict:
    """Execute a structured handoff with explicit context about the triggering finding."""
    handoff_message = (
        f"[HANDOFF FROM {from_agent_name.upper()}]\n"
        f"Trigger: {trigger_finding}\n"
        f"Question requiring your specialist assessment:\n{handoff_question}"
    )
    print(f"Handoff: {from_agent_name} -> {to_agent_name}")
    print(f"Trigger: {trigger_finding[:120]}")
    print()
    result = invoke_specialist(to_agent, handoff_message, agent_name=to_agent_name)
    return result


print("STEP 1: Finance Analyst analyses PFE and surfaces clinical flag")
print("─" * 65)
fa_initial = invoke_specialist(
    finance_analyst,
    "Retrieve current price and full fundamentals for PFE. Note any patent cliff events material to revenue outlook.",
    agent_name="Finance Analyst"
)
print_specialist_trace(fa_initial)
print(fa_initial["answer"][:500], "...\n")

print()
print("STEP 2: Handoff triggered — Clinical Risk takes over pipeline assessment")
print("─" * 65)
cr_handoff = execute_handoff(
    from_agent_name="Finance Analyst",
    to_agent=clinical_risk,
    to_agent_name="Clinical Risk",
    trigger_finding="PFE Eliquis patent expiry 2026 flagged as material revenue risk (~$6B annual). Pipeline offset assessment required.",
    handoff_question="Assess PFE's clinical pipeline for assets capable of partially offsetting the Eliquis revenue cliff by 2026-2028. "
                     "Include stage, indication, and risk assessment."
)
print_specialist_trace(cr_handoff)
print()
print("CLINICAL RISK HANDOFF RESPONSE:")
print(cr_handoff["answer"])

STEP 1: Finance Analyst analyses PFE and surfaces clinical flag
─────────────────────────────────────────────────────────────────
─────────────────────────────────────────────────────────────────
  Agent:    Finance Analyst
  ThreadID: 2f7945a2-5001-4956-a...
  Tokens:   in=703  out=243  total=946
  Steps:
    1. HumanMessage
    2. AIMessage -> ['get_stock_price', 'get_company_fundamentals']
    3. ToolMessage
    4. ToolMessage
    5. AIMessage
─────────────────────────────────────────────────────────────────
The current market price for Pfizer Inc. (PFE) is $29.54 USD. Below are the full fundamentals:

- **P/E Ratio**: 9.1
- **Market Capitalization**: $165 billion
- **Dividend Yield**: 6.20%
- **Sector**: Healthcare
- **Beta**: 0.62

I cannot comment on patent cliffs or revenue outlook as they fall outside the financial scope.

---

**FINANCIAL SUMMARY**  
- **Price**: $29.54 USD  
- **P/E**: 9.1  
- **Dividend Yield**: 6.20%  
- **Beta**: 0.62  
- **Investment View**: Pfizer offers

## Step 13: Fault Tolerance — Tool-Level Retry with Fallback
This step implements the retry and fallback patterns from Section 5.2 and 5.3. `with_retry` wraps any callable with configurable attempts and delay. `with_fallback` provides a safe default when all retries are exhausted. Two tests run: a flaky feed that recovers on attempt 3 (retry succeeds), and a permanently failing feed (fallback activates).

In [19]:
def with_retry(fn, *args, max_attempts: int = 3, delay: float = 0.5, **kwargs):
    """Execute fn with up to max_attempts attempts. Returns (result, attempts_taken, success)."""
    last_exc = None
    for attempt in range(1, max_attempts + 1):
        try:
            return fn(*args, **kwargs), attempt, True
        except Exception as e:
            last_exc = e
            print(f"  Attempt {attempt}/{max_attempts} failed: {type(e).__name__}: {e}")
            if attempt < max_attempts:
                time.sleep(delay)
    return None, max_attempts, False


def with_fallback(primary_fn, fallback_message: str, *args, max_attempts: int = 3, **kwargs):
    """Run primary_fn with retry. On exhaustion, return fallback_message."""
    result, attempts, success = with_retry(primary_fn, *args, max_attempts=max_attempts, **kwargs)
    if success:
        print(f"  Succeeded on attempt {attempts}.")
        return result
    print(f"  All {attempts} attempts failed. Activating fallback.")
    return fallback_message


_call_count = {"n": 0}

def flaky_price_feed(ticker: str) -> str:
    """Simulates a feed that fails twice then succeeds."""
    _call_count["n"] += 1
    if _call_count["n"] < 3:
        raise ConnectionError(f"Feed timeout on attempt {_call_count['n']} for {ticker}")
    return f"{ticker}: $29.54 USD (recovered after {_call_count['n']} attempts)"


print("Test 1: Retry succeeds on third attempt")
print("─" * 55)
_call_count["n"] = 0
result = with_fallback(
    flaky_price_feed,
    "Financial data unavailable for PFE. Recommend deferring to live data source.",
    "PFE",
    max_attempts=3
)
print(f"Result: {result}")

Test 1: Retry succeeds on third attempt
───────────────────────────────────────────────────────
  Attempt 1/3 failed: ConnectionError: Feed timeout on attempt 1 for PFE
  Attempt 2/3 failed: ConnectionError: Feed timeout on attempt 2 for PFE
  Succeeded on attempt 3.
Result: PFE: $29.54 USD (recovered after 3 attempts)


In [20]:
def always_failing_feed(ticker: str) -> str:
    raise ConnectionError(f"Feed permanently unavailable for {ticker}")


print("Test 2: All retries exhausted — fallback activates")
print("─" * 55)
result = with_fallback(
    always_failing_feed,
    "Financial data unavailable. Recommend deferring committee decision pending data restoration.",
    "XYZINVALID",
    max_attempts=3
)
print(f"Fallback result: {result}")

Test 2: All retries exhausted — fallback activates
───────────────────────────────────────────────────────
  Attempt 1/3 failed: ConnectionError: Feed permanently unavailable for XYZINVALID
  Attempt 2/3 failed: ConnectionError: Feed permanently unavailable for XYZINVALID
  Attempt 3/3 failed: ConnectionError: Feed permanently unavailable for XYZINVALID
  All 3 attempts failed. Activating fallback.
Fallback result: Financial data unavailable. Recommend deferring committee decision pending data restoration.


## Step 14: Agent-Level Fallback and Graceful Degradation
When a specialist fails completely, the Supervisor degrades gracefully rather than crashing. Finance Analyst failure produces a conservative financial caveat. Clinical Risk failure triggers a human-review escalation — a stronger response consistent with the higher safety threshold for clinical data described in Section 5.3. The degraded report clearly states which section is missing and why.

In [21]:
FINANCIAL_FALLBACK = (
    "FINANCIAL SUMMARY — DATA UNAVAILABLE\n"
    "The Finance Analyst specialist could not complete its analysis. "
    "Recommend deferring to a live market data source before committee decision."
)

CLINICAL_ESCALATION = (
    "CLINICAL RISK SUMMARY — ESCALATED TO HUMAN REVIEW\n"
    "The Clinical Risk specialist could not complete its analysis. "
    "Due to patient safety implications, this report must NOT be used for clinical decision-making. "
    "A qualified clinician must review before any action is taken."
)


def safe_finance_analysis(query: str) -> str:
    try:
        result = invoke_specialist(finance_analyst, query, "Finance Analyst")
        return result["answer"]
    except Exception as e:
        print(f"  Finance Analyst failed: {e}. Substituting fallback.")
        return FINANCIAL_FALLBACK


def safe_clinical_analysis(query: str) -> str:
    try:
        result = invoke_specialist(clinical_risk, query, "Clinical Risk")
        return result["answer"]
    except Exception as e:
        print(f"  Clinical Risk failed: {e}. Escalating to human review.")
        return CLINICAL_ESCALATION


def produce_committee_report(ticker: str, financial_output: str, clinical_output: str) -> str:
    prompt = (
        f"Produce a committee report for {ticker}. "
        f"If any section contains a data unavailability notice, reflect that caveat in your recommendation.\n\n"
        f"FINANCIAL ANALYST REPORT:\n{financial_output}\n\n"
        f"CLINICAL RISK REPORT:\n{clinical_output}"
    )
    return llm.invoke([HumanMessage(content=prompt)]).content


# Normal path — both specialists available
print("Graceful degradation — normal path:")
print("─" * 65)
fin = safe_finance_analysis("Price and fundamentals for ABBV.")
clin = safe_clinical_analysis("Pipeline risk assessment for ABBV.")
report = produce_committee_report("ABBV", fin, clin)
print(report[:700], "...\n" if len(report) > 700 else "\n")

Graceful degradation — normal path:
─────────────────────────────────────────────────────────────────
### Committee Report for AbbVie Inc. (ABBV)

#### Overview:
AbbVie Inc. (ABBV) is a biopharmaceutical company with a strong presence in immunology, oncology, and aesthetics. The company is currently navigating a critical transition period as it shifts its revenue base from its legacy blockbuster drug, Humira, to newer assets like Skyrizi and Rinvoq. This report evaluates AbbVie's financial performance, clinical risks, and overall investment outlook.

---

### Financial Analysis:
- **Stock Price:** $175.30 USD  
- **P/E Ratio:** 21.4 (moderate relative to the sector)  
- **Dividend Yield:** 3.85% (attractive for income-focused investors)  
- **Beta:** 0.71 (lower volatility compared to the br ...



In [22]:
# Degraded path — clinical specialist unavailable
print("Graceful degradation — clinical specialist unavailable:")
print("─" * 65)
fin_ok       = safe_finance_analysis("Price and fundamentals for ABBV.")
clin_fail    = CLINICAL_ESCALATION  # simulate failure by substituting directly
degraded_report = produce_committee_report("ABBV", fin_ok, clin_fail)
print(degraded_report)

Graceful degradation — clinical specialist unavailable:
─────────────────────────────────────────────────────────────────
### Committee Report for ABBV (AbbVie Inc.)

#### Executive Summary:
AbbVie Inc. (ABBV) presents a compelling investment opportunity based on its financial metrics, particularly for income-focused investors. However, clinical risk analysis remains incomplete, necessitating caution for stakeholders reliant on clinical data for decision-making. This report provides a balanced view of ABBV's financial and clinical standing, with recommendations reflecting the limitations of available data.

---

### Financial Analysis:
AbbVie demonstrates strong financial performance, as evidenced by the following metrics:
- **Price:** $175.30 USD
- **P/E Ratio:** 21.4, indicating moderate valuation relative to its sector.
- **Dividend Yield:** 3.85%, appealing to investors seeking consistent income.
- **Beta:** 0.71, suggesting lower volatility compared to the broader market.

**Recom

## Step 15: Human-in-the-Loop — Custom StateGraph with Interrupt
This step builds a custom `StateGraph` with a `human_review` node that calls `interrupt()`. The workflow: gather financial data -> gather clinical data -> pause for human review -> synthesise final report. The interrupt persists full state to the checkpointer. Resumption via `Command(resume=human_input)` incorporates the human directive into the synthesis step.

In [23]:
class CommitteeState(TypedDict):
    messages:          Annotated[list, add_messages]
    ticker:            str
    financial_output:  str
    clinical_output:   str
    human_directive:   str
    final_report:      str


def gather_financial(state: CommitteeState) -> CommitteeState:
    ticker = state["ticker"]
    print(f"  [Node: gather_financial] Analysing {ticker}...")
    output = safe_finance_analysis(
        f"Price, fundamentals, and dividend yield for {ticker}. Compare dividend yield to US 10-year Treasury."
    )
    return {"financial_output": output}


def gather_clinical(state: CommitteeState) -> CommitteeState:
    ticker = state["ticker"]
    print(f"  [Node: gather_clinical] Assessing {ticker} pipeline...")
    output = safe_clinical_analysis(f"Pipeline risk assessment and key regulatory risks for {ticker}.")
    return {"clinical_output": output}


def human_review(state: CommitteeState) -> CommitteeState:
    print(f"  [Node: human_review] Pausing for human review...")
    directive = interrupt({
        "financial_output": state["financial_output"],
        "clinical_output":  state["clinical_output"],
        "instruction": (
            "Review the specialist outputs above. Add a directive for the final report, "
            "or return 'APPROVED' to proceed without changes."
        )
    })
    return {"human_directive": directive}


def synthesise(state: CommitteeState) -> CommitteeState:
    ticker     = state["ticker"]
    directive  = state.get("human_directive", "APPROVED")
    print(f"  [Node: synthesise] Generating final report for {ticker}...")
    human_note = (
        f"Human directive: {directive}\n"
        if directive and directive.upper() != "APPROVED"
        else "Human reviewer approved without changes.\n"
    )
    prompt = (
        f"Produce the final committee report for {ticker}.\n{human_note}"
        f"FINANCIAL ANALYST OUTPUT:\n{state['financial_output']}\n\n"
        f"CLINICAL RISK OUTPUT:\n{state['clinical_output']}"
    )
    return {"final_report": llm.invoke([HumanMessage(content=prompt)]).content}


hitl_checkpointer   = MemorySaver()
hitl_graph_builder  = StateGraph(CommitteeState)
hitl_graph_builder.add_node("gather_financial", gather_financial)
hitl_graph_builder.add_node("gather_clinical",  gather_clinical)
hitl_graph_builder.add_node("human_review",     human_review)
hitl_graph_builder.add_node("synthesise",       synthesise)
hitl_graph_builder.add_edge(START,              "gather_financial")
hitl_graph_builder.add_edge("gather_financial", "gather_clinical")
hitl_graph_builder.add_edge("gather_clinical",  "human_review")
hitl_graph_builder.add_edge("human_review",     "synthesise")
hitl_graph_builder.add_edge("synthesise",       END)
hitl_graph = hitl_graph_builder.compile(checkpointer=hitl_checkpointer)

print("HITL committee graph compiled.")
print()
print("Graph structure (Mermaid):")
print(hitl_graph.get_graph().draw_mermaid())

HITL committee graph compiled.

Graph structure (Mermaid):
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	gather_financial(gather_financial)
	gather_clinical(gather_clinical)
	human_review(human_review)
	synthesise(synthesise)
	__end__([<p>__end__</p>]):::last
	__start__ --> gather_financial;
	gather_clinical --> human_review;
	gather_financial --> gather_clinical;
	human_review --> synthesise;
	synthesise --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [24]:
# Phase 1: Run up to the interrupt
hitl_session = {"configurable": {"thread_id": str(uuid.uuid4())}}

print("PHASE 1 — Running to human_review interrupt")
print("=" * 65)

initial_result = hitl_graph.invoke(
    {"ticker": "MRK", "financial_output": "", "clinical_output": "",
     "human_directive": "", "final_report": "", "messages": []},
    config=hitl_session
)

print()
print("Graph paused. Specialist outputs available for review:")
print("─" * 65)
print("FINANCIAL:\n", initial_result.get("financial_output", "")[:400], "...")
print()
print("CLINICAL:\n",  initial_result.get("clinical_output",  "")[:400], "...")

PHASE 1 — Running to human_review interrupt
  [Node: gather_financial] Analysing MRK...
  [Node: gather_clinical] Assessing MRK pipeline...
  [Node: human_review] Pausing for human review...

Graph paused. Specialist outputs available for review:
─────────────────────────────────────────────────────────────────
FINANCIAL:
 ### Financial Analysis for Merck & Co. (MRK)

#### Current Market Data:
- **Price**: $128.75 USD

#### Fundamentals:
- **P/E Ratio**: 18.2
- **Market Cap**: $325 billion USD
- **Dividend Yield**: 2.55%
- **Sector**: Healthcare
- **Beta**: 0.48 (low volatility compared to the market)

#### Benchmark Comparison:
- **US 10-Year Treasury Yield**: 4.48%
- **Dividend Yield vs. Treasury Yield**: MRK's di ...

CLINICAL:
 ### CLINICAL RISK SUMMARY

**Pipeline Stage:**  
Merck's pipeline is heavily concentrated on Keytruda, which has 21 approved indications and generates over $25 billion in annual revenue. The patent for Keytruda is set to expire in 2028, posing a significant 

In [25]:
# Phase 2: Human adds a directive and resumes
human_directive = (
    "Flag explicitly in the committee recommendation that MRK's Keytruda concentration risk is the primary "
    "concern for our long-term holding thesis. Add a monitoring trigger: if Keytruda revenue falls below "
    "60% of total pharma revenue in any quarter, escalate to committee review."
)

print("PHASE 2 — Resuming with human directive")
print("=" * 65)
print(f"Human directive: {human_directive[:150]}...")
print()

final_result = hitl_graph.invoke(
    Command(resume=human_directive),
    config=hitl_session
)

print()
print("FINAL COMMITTEE REPORT:")
print("=" * 65)
print(final_result.get("final_report", ""))

PHASE 2 — Resuming with human directive
Human directive: Flag explicitly in the committee recommendation that MRK's Keytruda concentration risk is the primary concern for our long-term holding thesis. Add a ...

  [Node: human_review] Pausing for human review...
  [Node: synthesise] Generating final report for MRK...

FINAL COMMITTEE REPORT:
### Final Committee Report for Merck & Co. (MRK)

#### Executive Summary:
Merck & Co. (MRK) remains a key holding within the healthcare sector, offering stable dividend income, low volatility, and exposure to oncology innovation. However, the committee has identified significant risks tied to the company's reliance on Keytruda, its flagship oncology drug, which accounts for a substantial portion of total pharmaceutical revenue. This concentration risk is the primary concern for our long-term holding thesis and necessitates heightened monitoring and strategic oversight.

---

#### Committee Recommendation:
The committee recommends maintaining MRK as

## Step 16: Parallel Specialist Invocation
When both specialist tasks are independent — neither needs the other's output to proceed — they run in parallel, reducing total wall-clock time. `ThreadPoolExecutor` runs both specialists concurrently while the Supervisor waits for both results before synthesising. This is the multi-agent equivalent of the parallel tool invocation pattern from the baseline lab.

In [26]:
def parallel_committee_analysis(ticker: str) -> dict:
    """Run Finance Analyst and Clinical Risk concurrently. Returns both outputs and elapsed time."""
    fin_query  = f"Price, fundamentals, dividend yield, and rate comparison for {ticker}."
    clin_query = f"Pipeline risk and regulatory risk assessment for {ticker}."
    start = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        fin_future  = executor.submit(invoke_specialist, finance_analyst,  fin_query,  "Finance Analyst")
        clin_future = executor.submit(invoke_specialist, clinical_risk,     clin_query, "Clinical Risk")
        fin_result  = fin_future.result()
        clin_result = clin_future.result()
    return {
        "financial_output":  fin_result["answer"],
        "clinical_output":   clin_result["answer"],
        "finance_tokens":    fin_result["token_usage"]["total"],
        "clinical_tokens":   clin_result["token_usage"]["total"],
        "elapsed_seconds":   round(time.time() - start, 2),
    }


print("Running Finance Analyst and Clinical Risk in parallel for PFE...")
print()
pr = parallel_committee_analysis("PFE")
print(f"Parallel execution completed in {pr['elapsed_seconds']}s")
print(f"Finance Analyst tokens:  {pr['finance_tokens']}")
print(f"Clinical Risk tokens:    {pr['clinical_tokens']}")
print()

final_parallel = produce_committee_report("PFE", pr["financial_output"], pr["clinical_output"])
print("FINAL COMMITTEE REPORT (parallel path):")
print("=" * 65)
print(final_parallel)

Running Finance Analyst and Clinical Risk in parallel for PFE...

Parallel execution completed in 6.8s
Finance Analyst tokens:  1129
Clinical Risk tokens:    966

FINAL COMMITTEE REPORT (parallel path):
### COMMITTEE REPORT: Pfizer Inc. (Ticker: PFE)

#### Executive Summary:
Pfizer Inc. (PFE) presents a mixed investment case, combining strong dividend yield and valuation metrics with clinical and revenue risks. The company’s fundamentals suggest it is undervalued, with a dividend yield of 6.20% that exceeds benchmark rates, making it attractive for income-focused investors. However, clinical pipeline uncertainties and looming patent expirations warrant caution. This report evaluates Pfizer’s financial and clinical outlook to provide a balanced recommendation.

---

### Financial Analysis:
**Key Metrics:**
- **Current Market Price**: $29.54 USD
- **P/E Ratio**: 9.1 (indicating undervaluation relative to earnings)
- **Dividend Yield**: 6.20% (higher than U.S. Treasury and Federal Funds r

## Step 17: Token Usage Across the Full Multi-Agent Stack
This step profiles token consumption at every layer: the Supervisor's outer loop, the Finance Analyst's inner loop, and the Clinical Risk agent's inner loop. The cross-agent total confirms the compounding cost of coordination described in Section 1.3 and validates the importance of scoped delegation and compact specialist outputs.

In [27]:
token_session = {"configurable": {"thread_id": str(uuid.uuid4())}, "recursion_limit": 20}

specialist_tokens = {}

# Wrap specialist invoke calls to capture inner loop token totals
_orig_fa_invoke  = finance_analyst.invoke
_orig_cr_invoke  = clinical_risk.invoke

def _tracked(original_fn, key):
    def wrapper(state, config=None):
        result = original_fn(state, config=config)
        total = sum(
            getattr(m, "response_metadata", {}).get("token_usage", {}).get("prompt_tokens", 0) +
            getattr(m, "response_metadata", {}).get("token_usage", {}).get("completion_tokens", 0)
            for m in result["messages"]
        )
        specialist_tokens[key] = specialist_tokens.get(key, 0) + total
        return result
    return wrapper

finance_analyst.invoke = _tracked(_orig_fa_invoke, "finance")
clinical_risk.invoke   = _tracked(_orig_cr_invoke, "clinical")

sv_result = supervisor_agent.invoke(
    {"messages": [HumanMessage(
        content="Produce an investment committee report on JNJ covering financial profile and clinical pipeline risk."
    )]},
    config=token_session
)

finance_analyst.invoke = _orig_fa_invoke
clinical_risk.invoke   = _orig_cr_invoke

sv_in, sv_out = 0, 0
for msg in sv_result["messages"]:
    usage = getattr(msg, "response_metadata", {}).get("token_usage", {})
    sv_in  += usage.get("prompt_tokens", 0)
    sv_out += usage.get("completion_tokens", 0)

fin_total   = specialist_tokens.get("finance",  0)
clin_total  = specialist_tokens.get("clinical", 0)
sv_total    = sv_in + sv_out
grand_total = sv_total + fin_total + clin_total

print("Token Usage Across the Multi-Agent Stack")
print("=" * 55)
print(f"  Supervisor (outer loop)       in={sv_in:>6}  out={sv_out:>5}  total={sv_total:>7}")
print(f"  Finance Analyst (inner loop)                            total={fin_total:>7}")
print(f"  Clinical Risk   (inner loop)                            total={clin_total:>7}")
print("-" * 55)
print(f"  Grand total                                             total={grand_total:>7}")
if sv_total > 0 and grand_total > 0:
    print(f"  Coordination overhead: {(fin_total + clin_total) / grand_total * 100:.1f}% of tokens consumed by specialist inner loops.")

    [inner loop] Finance Analyst | tokens=1062 | steps=5
    [inner loop] Clinical Risk | tokens=1546 | steps=6
Token Usage Across the Multi-Agent Stack
  Supervisor (outer loop)       in=  1411  out=  511  total=   1922
  Finance Analyst (inner loop)                            total=   1062
  Clinical Risk   (inner loop)                            total=   1546
-------------------------------------------------------
  Grand total                                             total=   4530
  Coordination overhead: 57.6% of tokens consumed by specialist inner loops.


## Summary: What was built and what it means
This lab constructed a complete multi-agent collaboration system through seventeen implementation steps, each grounded in one of the five theory sections.

**The ReAct Loop in a Multi-Agent Context** — The Supervisor, the Finance Analyst, and the Clinical Risk agent each run independent ReAct loops. The supervisor's loop treats specialist delegations as tool calls; specialist outputs are returned as `ToolMessage` objects, maintaining a consistent reasoning protocol across all three agents. Two-layer state management — outer session state in the supervisor's checkpointer, inner reasoning traces in each specialist's separate checkpointer — allows full auditability while keeping each agent's scope clean.

**Shared Memory and Message Passing** — Three memory scopes were implemented: session memory in the supervisor's `MemorySaver` checkpointer (Step 11), specialist memory isolated under separate `thread_id` namespaces (Steps 5 and 7), and scoped task messages as the shared workspace between agents (Step 10). Query scoping at the delegation boundary prevented context contamination between finance and clinical domains and reduced unnecessary token consumption in each specialist's inner loop.

**Supervisor and Handoff Patterns** — The supervisor pattern was implemented in Steps 8 and 9: the Supervisor decomposes queries, delegates to specialists via `@tool`-wrapped invocations, and synthesises a committee report. The handoff pattern was implemented in Step 12: the Finance Analyst raised a material patent cliff flag and explicitly transferred control to the Clinical Risk agent with a structured context message containing the triggering finding and the specific question. The Clinical Risk agent ran its full ReAct loop autonomously and returned a complete pipeline assessment.

**Human-in-the-Loop Control** — Step 15 built a custom `StateGraph` with a `human_review` node using `interrupt()`. Execution paused after both specialists completed but before synthesis. The human reviewer added a Keytruda concentration-risk monitoring trigger directive. The synthesis node incorporated it into the final report. The full interrupt-resume state is stored in the `MemorySaver` checkpointer and constitutes a durable, reproducible audit record of the review event.

**Fault Tolerance and Retry Logic** — Steps 13 and 14 implemented three fault layers: tool-level retry with configurable attempts and delay (`with_retry`), tool-level fallback with a safe default message (`with_fallback`), and agent-level graceful degradation where a failed Finance Analyst produces a conservative caveat and a failed Clinical Risk agent triggers human escalation rather than a silent omission. Step 17 quantified the token cost of the three-layer stack. Step 16 demonstrated parallel specialist invocation, running both agents concurrently when their tasks are independent, reducing wall-clock time while preserving the full audit trail for each agent's inner reasoning trace.